In [1]:
import sympy as sp
from helpers.einstein_helpers import minkowski_metric, euclidean_metric, ricci_tensor

# Parameters
D = 12
d = 2
HARMONIC_DIMENSIONS = D - d - 2

# Set up coordinates
x = sp.symbols(f"x0:{d}", real=True)
z = sp.symbols(f"z0:{2}", real=True)
y = sp.symbols(f"y0:{D - d - 2}", real=True)
coords = (*x, *z, *y)

# Parameters and symbols
r2 = sum(y_i**2 for y_i in y)
r = sp.sqrt(r2)
r_sym = sp.symbols('r', real=True, positive=True)

# harmonic function H, both as function of r, and as symbols
H = sp.Function('H')(r)
H_, H_p, H_pp = sp.symbols("H H' H''") # symbols for H and its derivatives, easier to work with once derivatives are taken and only manipulating algebraically

print(f"We are in {D} dimensions, with coordinates {coords}")
print(f"Radial coordinate r = {r}, r^2 = {r2}")
print(f"Harmonic function H is harmonic in {HARMONIC_DIMENSIONS} dimensions.")


# metric
g_MN = sp.zeros(D)
g_MN[0:d,0:d] = H**(sp.Rational(-3,4)) * minkowski_metric(d)
g_MN[d,d] = H**(sp.Rational(-1,2))
g_MN[d+1,d+1] = H**(sp.Rational(1,2))
g_MN[d+2:D,d+2:D] = H**(-sp.Rational(1, 4)) * euclidean_metric(D - d - 2)
g_MN = sp.simplify(g_MN)
g = sp.simplify(sp.sqrt(-(sp.det(g_MN))))
inv_g_MN = g_MN.inv()

print(f"Metric g_MN:\n{g_MN}")

# Compute the Ricci tensor
R_MN = ricci_tensor(g_MN, coords)

We are in 12 dimensions, with coordinates (x0, x1, z0, z1, y0, y1, y2, y3, y4, y5, y6, y7)
Radial coordinate r = sqrt(y0**2 + y1**2 + y2**2 + y3**2 + y4**2 + y5**2 + y6**2 + y7**2), r^2 = y0**2 + y1**2 + y2**2 + y3**2 + y4**2 + y5**2 + y6**2 + y7**2
Harmonic function H is harmonic in 8 dimensions.
Metric g_MN:
Matrix([[-1/H(sqrt(y0**2 + y1**2 + y2**2 + y3**2 + y4**2 + y5**2 + y6**2 + y7**2))**(3/4), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, H(sqrt(y0**2 + y1**2 + y2**2 + y3**2 + y4**2 + y5**2 + y6**2 + y7**2))**(-3/4), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1/sqrt(H(sqrt(y0**2 + y1**2 + y2**2 + y3**2 + y4**2 + y5**2 + y6**2 + y7**2))), 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, sqrt(H(sqrt(y0**2 + y1**2 + y2**2 + y3**2 + y4**2 + y5**2 + y6**2 + y7**2))), 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, H(sqrt(y0**2 + y1**2 + y2**2 + y3**2 + y4**2 + y5**2 + y6**2 + y7**2))**(-1/4), 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, H(sqrt(y0**2 + y1**2 + y2**2 + y3**2 + y4**2 + y5**2 + y6**2 + y7**2))**(-1/4), 0, 0, 0, 

In [2]:
def harmonic_derivative_subs(expr):
    # First sub in r and r^2
    expr = expr.subs({r: r_sym,
                      r2: r_sym**2})
    expr = sp.simplify(expr)
    expr = sp.simplify(expr.subs({sp.sqrt(r2): r_sym,
                                  r2: r_sym**2}))
    
    # Then sub symbols of H, H', H'' for its literal derivatives
    expr = sp.simplify(expr.subs({sp.Function('H')(r_sym): H_,
                                  sp.Derivative(sp.Function('H')(r_sym), r_sym): H_p,
                                  sp.Derivative(sp.Function('H')(r_sym), (r_sym, 2)): H_pp}))
    
    # now impose harmonic condition to eliminate H''
    expr = expr.subs({H_pp: -(HARMONIC_DIMENSIONS -1)/r_sym * H_p})
    return sp.simplify(expr)

In [3]:
R_00 = harmonic_derivative_subs(R_MN[0,0])
R_00 = sp.factor(R_00).subs(r**2, r_sym**2)
R_00

15*H'**2/(16*H**(5/2))

In [5]:
R_22 = harmonic_derivative_subs(R_MN[2,2])
R_22 = sp.factor(R_22).subs(r**2, r_sym**2)
R_22

-5*H'**2/(8*H**(9/4))

In [7]:
R_44 = harmonic_derivative_subs(R_MN[4,4])
R_44 = R_44.subs(sum([yi**2 for yi in y[1:]]), r_sym**2 - y[0]**2)
R_44

H'*(40*H*r**2 - 16*H*(r**2 + 12*y0**2) - H'*r*(31*y0**2 + 5*y1**2 + 5*y2**2 + 5*y3**2 + 5*y4**2 + 5*y5**2 + 5*y6**2 + 5*y7**2))/(16*H**2*r**3)